In [0]:
%pip install sentence-transformers pandas langchain langchain-community langchain-text-splitters
dbutils.library.restartPython()

In [0]:
import pandas as pd
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

# 1. Provide your Hugging Face Token
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "enter hugging face"

# 2. Load the BNS dataset directly from your workspace catalog
print("Loading table from workspace.default...")
try:
    # Reading CSV file from volume using spark.read.csv()
    spark_df = spark.read.csv("/Volumes/workspace/default/bns_dataset/bns_sections.csv", header=True, inferSchema=True)
    df = spark_df.toPandas()
    print(f"Successfully loaded table with {len(df)} sections.")
except Exception as e:
    print("ERROR: Could not find the table 'workspace.default.bns_dataset'.")
    raise e

# 3. The Indian Legal Chunking Logic
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600, 
    chunk_overlap=50,
    separators=["\n(1)", "\n(2)", "\n(3)", "\n(a)", "\n(b)", "\n\n", "\n", " ", ""] 
)

print("Chunking BNS Legal Text...")
chunked_data = []
for index, row in df.iterrows():
    if pd.isna(row['Description']):
        continue
        
    description = str(row['Description'])
    chunks = text_splitter.split_text(description)
    
    for chunk_id, chunk in enumerate(chunks):
        chunked_data.append({
            'Chapter': str(row['Chapter']),
            'Section': str(row['Section']),
            # Handling the space in your CSV's column name 'Section _name'
            'Section_name': str(row['Section _name']), 
            'Chunk_ID': f"SEC_{row['Section']}_CHUNK_{chunk_id}",
            'Text_Chunk': chunk
        })

chunked_df = pd.DataFrame(chunked_data)
print(f"Resulting smaller chunks for the AI: {len(chunked_df)}")

# 4. Initialize the "Made in India" Embedding Model
print("Downloading L3Cube Pune Indic-Sentence-BERT model...")
model = SentenceTransformer('l3cube-pune/indic-sentence-bert-nli')

# 5. Generate Vector Embeddings
print("Generating mathematical vectors for all legal chunks (This will take a minute or two)...")
chunked_df['Embedding_Vector'] = chunked_df['Text_Chunk'].apply(lambda x: model.encode(str(x)).tolist())

# 6. Save to Delta Table in your workspace catalog
print("Saving to Vector Table in workspace.default...")
final_spark_df = spark.createDataFrame(chunked_df)

# Saving it to your exact, approved Unity Catalog location
table_name = "workspace.default.bns_vector_table" 
final_spark_df.write.format("delta").mode("overwrite").saveAsTable(table_name)

print(f"SUCCESS! Data embedded and saved to Delta Table: {table_name}")